In [1]:
import numpy as np 
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

In [2]:
titanic=pd.read_csv("/kaggle/input/datasets/fatuzahra/titanic/titanic.csv")
titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
cabin_dropped= titanic.drop(columns="Cabin")

In [4]:
cabin_dropped["Age"].fillna(
    cabin_dropped["Age"].mean(),inplace=True
)

/tmp/ipykernel_17/4064252882.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  cabin_dropped["Age"].fillna(


In [5]:
cabin_dropped["Embarked"].fillna(
    cabin_dropped["Embarked"].mode()[0], inplace=True
)
cabin_dropped.isnull().sum()

/tmp/ipykernel_17/3271196257.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  cabin_dropped["Embarked"].fillna(


PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64

In [6]:
Q1=cabin_dropped["Fare"].quantile(0.25) #as it's the 25th percentile
Q3=cabin_dropped["Fare"].quantile(0.75) #as it's the 75th percentile

print(Q1)
print(Q3)

IQR=Q3-Q1

print(IQR)

lower_bound=Q1-1.5*IQR
upper_bound=Q3+1.5*IQR

print(lower_bound)
print(upper_bound)

titanic_clean=cabin_dropped[
(cabin_dropped["Fare"]>=lower_bound)&(cabin_dropped["Fare"]<=upper_bound)
]

#now we can visibly see that the outlier data was dropped

titanic_clean.shape

7.9104
31.0
23.0896
-26.724
65.6344


(775, 11)

In [7]:
titanic=titanic[['Survived','Pclass','Age','Fare','Sex']]

titanic['Age'].fillna(titanic['Age'].median(),inplace=True)

/tmp/ipykernel_17/1787994337.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  titanic['Age'].fillna(titanic['Age'].median(),inplace=True)


In [8]:
titanic['Sex']=titanic['Sex'].map({'male':0,'female':1})

In [9]:
X= titanic.drop('Survived',axis=1)
Y=titanic['Survived']

In [10]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42
)

In [11]:
baseline_model= DummyClassifier(strategy='most_frequent')
baseline_model.fit(X_train,Y_train)

DummyClassifier(strategy='most_frequent')

In [12]:
model=LogisticRegression(max_iter=1000)

In [13]:
model.fit(X_train,Y_train)

LogisticRegression(max_iter=1000)

In [14]:
print("Weights:",model.coef_)
print("Bias",model.intercept_)

Weights: [[-9.96364826e-01 -2.51645337e-02  1.18224793e-03  2.46563064e+00]]
Bias [1.50122817]


In [15]:
y_pred=model.predict(X_test)

y_prob=model.predict_proba(X_test)

In [16]:
print("Predictions",y_pred[:5])
print("Probabilities:",y_prob[:5])

Predictions [0 0 0 1 1]
Probabilities: [[0.89793144 0.10206856]
 [0.77888637 0.22111363]
 [0.8788716  0.1211284 ]
 [0.13445193 0.86554807]
 [0.34552886 0.65447114]]


In [17]:
from sklearn.metrics import accuracy_score, confusion_matrix,classification_report

In [18]:
accuracy=accuracy_score(Y_test,y_pred)
print("Accuracy:",accuracy)

Accuracy: 0.8044692737430168


In [19]:
cm=confusion_matrix(Y_test,y_pred)
print("Confusion matrix:",cm)

Confusion matrix: [[90 15]
 [20 54]]


In [20]:
c_report=classification_report(Y_test,y_pred)
print("Classification report:",c_report)

Classification report:               precision    recall  f1-score   support

           0       0.82      0.86      0.84       105
           1       0.78      0.73      0.76        74

    accuracy                           0.80       179
   macro avg       0.80      0.79      0.80       179
weighted avg       0.80      0.80      0.80       179



In [21]:
predictions=baseline_model.predict(X_test)
baseline_accuracy=accuracy_score(Y_test,predictions)
print("Baseline Model Accuracy:",baseline_accuracy)

Baseline Model Accuracy: 0.5865921787709497


In [22]:
y_train_pred=model.predict(X_train)
test_accuracy=accuracy_score(Y_train,y_train_pred)
print("Test accuracy:",test_accuracy)

Test accuracy: 0.7907303370786517
